# HighRes MicroServe

```{device-card} highres-microserve
```

| Property | Value |
| --- | --- |
| Transport | Ethernet, TCP port 1000 |
| Storage | 14 stackers, indexed 0–13 |
| Direct connection address | `10.253.253.253` |
| Verified controller | HRB-2008-10558, firmware 2.7.0.756 |
| Public length units | millimeters |

**Plate transfers, barcode scanning, and physical fault recovery remain unverified.** Communication, homing, all fourteen carousel positions, and empty receiving-position preparation/retraction were checked on this controller. Review the physical setup before running each motion cell.

This driver prepares the MicroServe for a separate robot to place or pick plates. It does not model plate inventory in PyLabRobot resources. Reported plate counts are the controller's cached estimates.

See [Protocol and validation](protocol.md) for the device-source mapping and verification limits.

## Connection and manufacturer documentation

Connect the powered MicroServe's Ethernet port to a dedicated NIC. Configure that NIC with `10.253.253.50/24`, then use `10.253.253.253` as the host. All HighRes controllers share that alternate address, so use it only on a dedicated link. Your normal network connection can remain separate.

The controller serves its [MicroServe API PDF](http://10.253.253.253/support_files/files/MicroServe_API.pdf) through its [Support Files page](http://10.253.253.253/support_files.html). This guide follows revision 756, dated May 29, 2020. The saved network address shown by the web interface can differ from legacy in-memory `settings` values.

The protocol acknowledges a command with an identifier, sends data, then sends a completion with the same command and identifier. The driver handles framing and converts plate dimensions from millimeters to device micrometers.

## Physical setup

Secure the machine, keep the robot outside its motion area, and inspect the transfer position. Each stacker must contain only one compatible plate geometry. Measure plate height, stacking pitch, and spatula support thickness before moving plates. Resolve an active E-stop through the machine's normal procedure.

## Connect

`setup()` opens the socket without homing, clearing errors, or changing calibration.

In [ ]:
from pylabrobot.high_res import HighResMicroServe, MicroServePlateDimensions

microserve = HighResMicroServe(host="10.253.253.253")
await microserve.setup()

## Identification

Read the controller identification report.

In [ ]:
print(await microserve.request_version())

## Firmware version

Read the machine-readable firmware version.

In [ ]:
await microserve.request_firmware_version()

## Status

Inspect homing, selected stacker, loader state, and the plate-detection beam. `plate_sensor_blocked` reports the beam signal; it does not establish plate occupancy at the virtual transfer position. An unknown stacker before homing is `None`.

Use `status.loader_retracted` and `status.loader_extended` to check the loader sensors. The raw `status.loader` field can stay `"extended"` after retraction on this firmware.

In [ ]:
status = await microserve.request_status()
status

## Readiness

This reports readiness to present a plate to the robot. It is false while the loader is retracted, even when the machine is homed and idle. Use `request_status()` to check homing and busy state.

In [ ]:
await microserve.is_ready()

## Error log

Read error entries without clearing them. Their numbers identify log entries, not stable fault codes.

In [ ]:
await microserve.request_errors()

## Settings

Read the current in-memory settings without changing or saving them.

In [ ]:
settings = await microserve.request_settings()
settings["PRODUCT_NAME"]

## Current plate geometry

Read all stackers' configured dimensions in millimeters.

In [ ]:
await microserve.request_dimensions()

## Cached plate counts

These are approximate firmware counts. This query does not move the carousel or physically recount plates.

In [ ]:
await microserve.request_plate_counts()

## Detailed hardware versions

Read firmware checksums and motor-controller versions.

In [ ]:
print(await microserve.request_detailed_version())

## Motor identities

Read the serial number and configured name of each axis controller.

In [ ]:
await microserve.request_motor_information()

## Command history

Read the latest three command history entries without repeating any command.

In [ ]:
await microserve.request_history(3)

## Command completion record

Query the last acknowledged command. IDs are only valid until the controller restarts.

In [ ]:
command_id = microserve.last_command_id
assert command_id is not None
await microserve.request_command_status(command_id)

## One stacker

Read a single stacker's cached approximate count without measurement motion.

In [ ]:
await microserve.stackers[0].request_plate_count()

## Home

**Moves hardware.** Clear the robot and transfer position first. Homing is explicit. A repeated call checks status and does not home again when already homed.

In [ ]:
await microserve.home()

## Plate geometry

Use measurements for the plates in this stacker. The values below are examples, not a plate-type recommendation:

- `height`: bottom of plate to top.
- `stack_height`: bottom-to-bottom distance between adjacent stacked plates.
- `thickness`: top of plate to the underside of its wells where the spatula supports it.

Each prepare or scan call receives geometry explicitly.

In [ ]:
dimensions = MicroServePlateDimensions(height=11.0, stack_height=10.0, thickness=10.0)
stacker = microserve.stackers[0]

## Set dimensions

Apply geometry for this stacker and verify it by reading it back. Preparation and scan methods also perform this step.

In [ ]:
await stacker.set_dimensions(dimensions)

## Rotate to a stacker

**Moves hardware.** The loader must be retracted and the transfer position clear.

In [ ]:
await stacker.move_to()

## Prepare to load a plate

**Moves hardware.** Present an empty receiving position for the robot. This command does not move the robot. Repeating it in the same verified state does not move the loader again.

In [ ]:
await stacker.prepare_for_load(dimensions)

## Finish loading

Have the robot place the plate at the taught transfer position, then withdraw completely. Only after that physical action, retract the mechanism.

In [ ]:
await microserve.retract()

## Prepare to unload a plate

**Moves hardware.** Present a plate from this stacker for the robot to pick. The method confirms the command completed and the selected stacker and loader reached the expected state. A repeated preparation owned by this driver does not fetch another plate before retraction, even if the beam signal changes. Actual robot pickup geometry is unverified.

In [ ]:
await stacker.prepare_for_unload(dimensions)

## Finish unloading

Have the robot pick the presented plate and withdraw completely before running this cell.

In [ ]:
await microserve.retract()

## Scan stacker barcodes

**Moves hardware.** The robot must be clear and the loader retracted. The returned tuple contains the controller's unmodified barcode data lines; barcode formatting has not yet been verified on this machine.

In [ ]:
await stacker.scan_barcodes(dimensions)

## Failures and reconnecting

`MicroServeError` preserves the command, identifier, completion status, and diagnostic lines for `ERROR`, `ABORTED`, and `WARNING` replies. All three interrupt the operation.

A timeout, cancelled task, or malformed reply closes the connection. The driver never automatically repeats a command. Closing the socket does not stop motion already executing. Inspect the physical machine before reconnecting with `setup()` and reading status.

An interrupted load/unload is retained in `unresolved_preparation`, blocking further preparation. The beam signal alone cannot identify which transfer completed. If the controller has not rebooted and no other client has moved it, `reconcile_preparation()` reads the original command record and current position to confirm success. Failed or unidentifiable commands stay unresolved. Explicit retraction ends a handoff only after the operator has inspected it and the robot is clear.

## Inspect an interrupted preparation

This is `None` when no load/unload is unresolved.

In [ ]:
microserve.unresolved_preparation

## Reconcile without moving

Use only after inspecting the machine and confirming it has not rebooted or been moved by another client. This reads the command record and loader state; it is a no-op when nothing is unresolved.

In [ ]:
await microserve.reconcile_preparation()

## Disconnect

Close the connection without homing, retracting, aborting, or clearing errors.

In [ ]:
await microserve.stop()